# Serialization & Persistence

## Overview

Once you've defined and sampled configurations, you need to **save and load them**. SpaX supports multiple serialization formats out of the box.

**What you'll learn:**
- Serializing configs to JSON, YAML, and TOML
- Deserializing (loading) configs from strings
- How nested configs are handled in serialization
- Error handling when deserialization fails
- Choosing the right format for your use case

**Prerequisites:**
- Basic understanding of SpaX `Config` (see notebook 00)
- Familiarity with nested configs (helpful, see notebook 02)

**Why serialization matters:**
- 📝 **Experiment tracking**: Save exact config used for each run
- 🔄 **Reproducibility**: Load config to reproduce results exactly
- 👥 **Sharing**: Send configs to teammates or version control
- 🐛 **Debugging**: Inspect configs that caused issues

**Supported formats:**
- **JSON**: Universal, great for APIs and tooling
- **YAML**: Human-readable, popular in ML (requires `PyYAML`)
- **TOML**: Clean syntax, gaining popularity (requires `tomli-w`)

Let's start with the simplest format: JSON.

In [ ]:
# Import SpaX
import spax as sp


# Simple config for demonstration
class ModelConfig(sp.Config):
    """Simple model configuration."""

    num_layers: int = sp.Int(ge=1, le=10)
    hidden_dim: int = sp.Int(ge=64, le=512)
    learning_rate: float = sp.Float(ge=1e-5, le=1e-1, distribution="log")
    optimizer: str = sp.Categorical(["adam", "sgd", "rmsprop"])
    use_dropout: bool = sp.Categorical([True, False])


# Create a config instance
config = ModelConfig.random(seed=42)
print("📦 Original config:")
print(config)
print("\n" + "=" * 60 + "\n")

# Serialize to JSON
json_string = config.model_dump_json()
print("📄 JSON serialization:")
print(json_string)
print("\n" + "=" * 60 + "\n")

# Deserialize from JSON
loaded_config = ModelConfig.model_validate_json(json_string)
print("✅ Loaded config:")
print(loaded_config)
print("\n" + "=" * 60 + "\n")

# Verify they're identical
print("🔍 Verification:")
print(f"  num_layers match: {config.num_layers == loaded_config.num_layers}")
print(f"  learning_rate match: {config.learning_rate == loaded_config.learning_rate}")
print(f"  optimizer match: {config.optimizer == loaded_config.optimizer}")
print("\n✨ Serialization preserves all values perfectly!")

📦 Original config:
ModelConfig(num_layers=2, hidden_dim=76, learning_rate=0.009251283695699343, optimizer='adam', use_dropout=True)


📄 JSON serialization:
{
  "num_layers": 2,
  "hidden_dim": 76,
  "learning_rate": 0.009251283695699343,
  "optimizer": "adam",
  "use_dropout": true
}


✅ Loaded config:
ModelConfig(num_layers=2, hidden_dim=76, learning_rate=0.009251283695699343, optimizer='adam', use_dropout=True)


🔍 Verification:
  num_layers match: True
  learning_rate match: True
  optimizer match: True

✨ Serialization preserves all values perfectly!


In [ ]:
# YAML serialization (requires PyYAML)
try:
    yaml_string = config.model_dump_yaml()
    print("📘 YAML serialization:")
    print(yaml_string)
    print("=" * 60 + "\n")

    # Deserialize from YAML
    loaded_from_yaml = ModelConfig.model_validate_yaml(yaml_string)
    print("✅ Loaded from YAML:")
    print(f"  optimizer: {loaded_from_yaml.optimizer}")
    print(f"  learning_rate: {loaded_from_yaml.learning_rate:.6f}")

except RuntimeError as e:
    print("⚠️  YAML not available:")
    print(f"   {e}")
    print("   Install with: pip install PyYAML\n")

print("\n" + "=" * 60 + "\n")

# TOML serialization (requires tomli-w)
try:
    toml_string = config.model_dump_toml()
    print("📗 TOML serialization:")
    print(toml_string)
    print("=" * 60 + "\n")

    # Deserialize from TOML
    loaded_from_toml = ModelConfig.model_validate_toml(toml_string)
    print("✅ Loaded from TOML:")
    print(f"  optimizer: {loaded_from_toml.optimizer}")
    print(f"  learning_rate: {loaded_from_toml.learning_rate:.6f}")

except RuntimeError as e:
    print("⚠️  TOML not available:")
    print(f"   {e}")
    print("   Install with: pip install tomli-w\n")

📘 YAML serialization:
num_layers: 2
hidden_dim: 76
learning_rate: 0.009251283695699343
optimizer: adam
use_dropout: true


✅ Loaded from YAML:
  optimizer: adam
  learning_rate: 0.009251


📗 TOML serialization:
num_layers = 2
hidden_dim = 76
learning_rate = 0.009251283695699343
optimizer = "adam"
use_dropout = true


✅ Loaded from TOML:
  optimizer: adam
  learning_rate: 0.009251


## 📊 Format Comparison

Each format has different strengths:

| Format | Pros | Cons | Best For |
|--------|------|------|----------|
| **JSON** | ✅ Universal<br>✅ No dependencies<br>✅ Great for APIs | ❌ Less human-readable<br>❌ No comments | Programmatic use, APIs, tooling |
| **YAML** | ✅ Very readable<br>✅ Supports comments<br>✅ Popular in ML | ❌ Requires PyYAML<br>❌ Indentation-sensitive | Config files, human editing |
| **TOML** | ✅ Clean syntax<br>✅ Less error-prone<br>✅ Growing adoption | ❌ Requires tomli-w<br>❌ Less common | Modern config files |

**General recommendation:**
- **Development/sharing**: YAML (most readable)
- **Production/APIs**: JSON (universal)
- **Modern projects**: TOML (clean and robust)

---

## 🏗️ Nested Configs: The Tricky Part

When configs contain other configs, SpaX needs to track **which config type** each nested field is. This is handled automatically with `__type__` discriminators.

Let's see how it works:

In [ ]:
# Nested configs require type information for deserialization
class OptimizerConfig(sp.Config):
    """Optimizer configuration."""

    name: str = sp.Categorical(["adam", "sgd"])
    learning_rate: float = sp.Float(ge=1e-5, le=1e-2, distribution="log")


class TrainingConfig(sp.Config):
    """Training configuration with nested optimizer."""

    optimizer: OptimizerConfig  # Nested config
    batch_size: int = sp.Int(ge=16, le=128)
    num_epochs: int = sp.Int(ge=1, le=50)


# Create and serialize
training_config = TrainingConfig.random(seed=100)
print("📦 Original nested config:")
print(training_config)
print("\n" + "=" * 60 + "\n")

# Serialize to JSON - notice the __type__ field!
json_str = training_config.model_dump_json()
print("📄 JSON with nested config:")
print(json_str)
print("\n" + "=" * 60 + "\n")

# Deserialize - SpaX uses __type__ to reconstruct the correct type
loaded = TrainingConfig.model_validate_json(json_str)
print("✅ Loaded nested config:")
print(loaded)
print("\n" + "=" * 60 + "\n")

# Verify nested fields work correctly
print("🔍 Nested field verification:")
print(f"  Original optimizer name: {training_config.optimizer.name}")
print(f"  Loaded optimizer name: {loaded.optimizer.name}")
print(f"  Types match: {type(training_config.optimizer) is type(loaded.optimizer)}")
print(f"  Values match: {training_config.optimizer.name == loaded.optimizer.name}")

📦 Original nested config:
TrainingConfig(optimizer=OptimizerConfig(name='adam', learning_rate=0.0002316226433137286), batch_size=114, num_epochs=12)


📄 JSON with nested config:
{
  "optimizer": {
    "name": "adam",
    "learning_rate": 0.0002316226433137286,
    "__type__": "OptimizerConfig"
  },
  "batch_size": 114,
  "num_epochs": 12
}


✅ Loaded nested config:
TrainingConfig(optimizer=OptimizerConfig(name='adam', learning_rate=0.0002316226433137286), batch_size=114, num_epochs=12)


🔍 Nested field verification:
  Original optimizer name: adam
  Loaded optimizer name: adam
  Types match: True
  Values match: True


In [ ]:
# Polymorphic configs: Union types need __type__ to know which variant was used
class AdamConfig(sp.Config):
    """Adam optimizer."""

    learning_rate: float = sp.Float(ge=1e-5, le=1e-2, distribution="log")
    beta1: float = sp.Float(ge=0.8, le=0.99)
    beta2: float = sp.Float(ge=0.9, le=0.999)


class SGDConfig(sp.Config):
    """SGD optimizer."""

    learning_rate: float = sp.Float(ge=1e-4, le=1e-1, distribution="log")
    momentum: float = sp.Float(ge=0.0, le=0.99)


class FlexibleTrainingConfig(sp.Config):
    """Training config with polymorphic optimizer field."""

    optimizer: AdamConfig | SGDConfig  # Can be either type!
    batch_size: int = sp.Int(ge=16, le=128)


# Sample multiple times to get different optimizer types
print("🎲 Serializing different optimizer types:\n")

for seed in [200, 201, 202]:
    config = FlexibleTrainingConfig.random(seed=seed)
    optimizer_type = type(config.optimizer).__name__

    # Serialize
    json_str = config.model_dump_json(indent=None)  # Compact for display

    print(f"Seed {seed}: {optimizer_type}")
    print(f"  JSON: {json_str[:80]}...")

    # Deserialize and verify type is preserved
    loaded = FlexibleTrainingConfig.model_validate_json(json_str)
    loaded_type = type(loaded.optimizer).__name__
    print(f"  Loaded type: {loaded_type} ✅")
    print()

print("=" * 60)
print("\n💡 The __type__ field tells SpaX which config class to instantiate!")

🎲 Serializing different optimizer types:

Seed 200: AdamConfig
  JSON: {"optimizer": {"learning_rate": 4.0770020008354014e-05, "beta1": 0.9347335569990...
  Loaded type: AdamConfig ✅

Seed 201: AdamConfig
  JSON: {"optimizer": {"learning_rate": 9.479788825955983e-05, "beta1": 0.80069832025593...
  Loaded type: AdamConfig ✅

Seed 202: SGDConfig
  JSON: {"optimizer": {"learning_rate": 0.008596907487459896, "momentum": 0.405343699338...
  Loaded type: SGDConfig ✅


💡 The __type__ field tells SpaX which config class to instantiate!


In [ ]:
# Error handling: What happens when deserialization fails?
print("🚨 Error Handling Examples:\n")

# Error 1: Invalid JSON syntax
print("1️⃣  Invalid JSON syntax:")
try:
    bad_json = '{"num_layers": 5, "hidden_dim": 128'  # Missing closing brace
    ModelConfig.model_validate_json(bad_json)
except Exception as e:
    print(f"   ❌ {type(e).__name__}: {str(e)[:80]}...")
    print()

# Error 2: Value outside space bounds
print("2️⃣  Value outside allowed range:")
try:
    bad_value = '{"num_layers": 999, "hidden_dim": 128, "learning_rate": 0.001, "optimizer": "adam", "use_dropout": true}'
    ModelConfig.model_validate_json(bad_value)
except Exception as e:
    print(f"   ❌ {type(e).__name__}")
    print("   Reason: num_layers=999 is outside Int([1, 10])")
    print()

# Error 3: Invalid choice for categorical
print("3️⃣  Invalid categorical choice:")
try:
    bad_choice = '{"num_layers": 5, "hidden_dim": 128, "learning_rate": 0.001, "optimizer": "invalid_opt", "use_dropout": true}'
    ModelConfig.model_validate_json(bad_choice)
except Exception as e:
    print(f"   ❌ {type(e).__name__}")
    print("   Reason: 'invalid_opt' not in ['adam', 'sgd', 'rmsprop']")
    print()

# Error 4: Missing required field
print("4️⃣  Missing required field:")
try:
    missing_field = '{"num_layers": 5, "hidden_dim": 128}'  # Missing learning_rate, optimizer, use_dropout
    ModelConfig.model_validate_json(missing_field)
except Exception as e:
    print(f"   ❌ {type(e).__name__}")
    print("   Reason: Required fields are missing")
    print()

# Error 5: Wrong nested config type (for polymorphic fields)
print("5️⃣  Wrong __type__ in nested config:")
try:
    wrong_type = '{"optimizer": {"learning_rate": 0.001, "momentum": 0.9, "__type__": "NonExistentConfig"}, "batch_size": 32}'
    FlexibleTrainingConfig.model_validate_json(wrong_type)
except Exception as e:
    print(f"   ❌ {type(e).__name__}")
    print("   Reason: Config type 'NonExistentConfig' not found")
    print()

print("=" * 60)
print("\n✅ SpaX validates thoroughly during deserialization!")
print("   Always check that loaded configs match your expectations.")

🚨 Error Handling Examples:

1️⃣  Invalid JSON syntax:
   ❌ JSONDecodeError: Expecting ',' delimiter: line 1 column 36 (char 35)...

2️⃣  Value outside allowed range:
   ❌ ValidationError
   Reason: num_layers=999 is outside Int([1, 10])

3️⃣  Invalid categorical choice:
   ❌ ValidationError
   Reason: 'invalid_opt' not in ['adam', 'sgd', 'rmsprop']

4️⃣  Missing required field:
   ❌ RuntimeError
   Reason: Required fields are missing

5️⃣  Wrong __type__ in nested config:
   ❌ ValueError
   Reason: Config type 'NonExistentConfig' not found


✅ SpaX validates thoroughly during deserialization!
   Always check that loaded configs match your expectations.


In [ ]:
# Practical workflow: Save before experiment, load to reproduce
print("🔬 Practical Experiment Workflow:\n")

# Step 1: Create and save config before training
print("Step 1: Generate and save config before experiment")
experiment_config = ModelConfig.random(seed=999)
config_json = experiment_config.model_dump_json()

# In real workflow, you'd save to file:
# with open('experiment_config.json', 'w') as f:
#     f.write(config_json)

print("  ✅ Config saved (seed=999)")
print(
    f"     optimizer={experiment_config.optimizer}, lr={experiment_config.learning_rate:.6f}"
)
print()

# Step 2: Run experiment (simulated)
print("Step 2: Run experiment with this config")
print("  🏃 Training model...")
print("  📊 Final accuracy: 94.5%")
print()

# Step 3: Later, load config to reproduce results
print("Step 3: Load config to reproduce results")
# In real workflow:
# with open('experiment_config.json', 'r') as f:
#     config_json = f.read()

reproduced_config = ModelConfig.model_validate_json(config_json)
print("  ✅ Config loaded")
print(
    f"     optimizer={reproduced_config.optimizer}, lr={reproduced_config.learning_rate:.6f}"
)
print()

# Verify exact match
print("Step 4: Verify exact reproduction")
print(
    f"  Configs match: {experiment_config.num_layers == reproduced_config.num_layers}"
)
print(
    f"  Optimizer match: {experiment_config.optimizer == reproduced_config.optimizer}"
)
print(
    f"  LR match: {experiment_config.learning_rate == reproduced_config.learning_rate}"
)
print()
print("  ✅ Experiment is perfectly reproducible!")

print("\n" + "=" * 60)
print("\n💡 Best practices:")
print("   • Save config BEFORE running experiment")
print("   • Use version control for config files")
print("   • Include config hash/timestamp in experiment logs")
print("   • Load config when reproducing or debugging")

🔬 Practical Experiment Workflow:

Step 1: Generate and save config before experiment
  ✅ Config saved (seed=999)
     optimizer=sgd, lr=0.001867

Step 2: Run experiment with this config
  🏃 Training model...
  📊 Final accuracy: 94.5%

Step 3: Load config to reproduce results
  ✅ Config loaded
     optimizer=sgd, lr=0.001867

Step 4: Verify exact reproduction
  Configs match: True
  Optimizer match: True
  LR match: True

  ✅ Experiment is perfectly reproducible!


💡 Best practices:
   • Save config BEFORE running experiment
   • Use version control for config files
   • Include config hash/timestamp in experiment logs
   • Load config when reproducing or debugging


## 📝 Summary: Serialization & Persistence

You've learned how to save and load SpaX configurations in multiple formats:

### ✅ Three Serialization Formats

| Format | Method | Requires |
|--------|--------|----------|
| **JSON** | `model_dump_json()` / `model_validate_json()` | Nothing (built-in) |
| **YAML** | `model_dump_yaml()` / `model_validate_yaml()` | `pip install PyYAML` |
| **TOML** | `model_dump_toml()` / `model_validate_toml()` | `pip install tomli-w` |

### ✅ Key Concepts

1. **Simple configs** serialize naturally to all formats
2. **Nested configs** use `__type__` discriminators to preserve structure
3. **Polymorphic fields** (Union types) use `__type__` to track which variant was used
4. **Validation** happens during deserialization - invalid values are caught immediately
5. **Reproducibility** is guaranteed - serialized configs preserve exact values

### 🎯 When to Use Each Format

- **JSON**: APIs, tooling, universal compatibility (no dependencies)
- **YAML**: Config files, human editing, comments support
- **TOML**: Modern projects, clean syntax, less error-prone

### 🚨 Common Errors

SpaX catches these issues during deserialization:
- ❌ Malformed syntax (invalid JSON/YAML/TOML)
- ❌ Values outside space bounds
- ❌ Invalid categorical choices
- ❌ Missing required fields
- ❌ Unknown config types in `__type__` fields

### 🔬 Practical Workflow
```python
# 1. Save before experiment
config = MyConfig.random(seed=42)
json_str = config.model_dump_json()
# with open('config.json', 'w') as f: f.write(json_str)

# 2. Load to reproduce
# with open('config.json', 'r') as f: json_str = f.read()
loaded = MyConfig.model_validate_json(json_str)
```

### 🚀 What's Next?
- **Notebook 04**: HPO with Optuna integration
- **Notebook 05**: Iterative refinement with overrides

**Your configs are now persistent and reproducible! 🎉**